# Pipeline v10 — VTIT LLM RAG Contest

**Mục tiêu**: acc ≥70%, time ≤10s/câu trên T4 Colab.

**Stack**:
- LLM: Qwen2.5-7B-Instruct-AWQ qua HF Transformers + SDPA (KHÔNG dùng vLLM vì T4 sm_75)
- Embedding: BGE-M3 fine-tuned (đã có)
- Retrieval: Qdrant local + BM25 (optional)
- Router: rule-based + SVM (bỏ LLM tiebreaker để tiết kiệm time)

**Key changes vs v8**:
1. `call_document` dùng **MCQA via logprob** (1 forward, không generate text)
2. `call_api` dùng **deterministic extractor** cho time/org/projectType, LLM chỉ extract phần còn thiếu
3. Router 3 tầng: hard-rule → SVM → (bỏ LLM tiebreaker)
4. Pre-compute API embedding, share query embedding giữa các layer
5. Loại bỏ hack `{"body": ` prepend + `find('{')` parsing

**Cách chạy Pipeline**:

1.   chạy đến phần 2 load model qwen từ locall auto lỗi
2.   Vào restartthời gian chạy xong rồi chạy lại cell thứ nhất rồi cell chứa các đường dẫn , rồi cell bị lỗi đấy  là xong , tiếp tục chạy bình thường

## 1. Setup & mount drive

In [ ]:
# Run once. Comment out sau khi cài xong.
!pip install -q autoawq qdrant-client rank_bm25 sentence-transformers pyvi openpyxl accelerate

In [ ]:
!pip install -q gptqmodel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.1/979.1 kB 19.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.8/121.8 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 10.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR        = '/content/drive/MyDrive/Workspace_Chatbot_LLM_Output'
SVM_PATH        = f'{BASE_DIR}/models/svm_router_v1.0.pkl'
BGE_PATH        = f'{BASE_DIR}/models/bge-m3-finetuned-offline'
DB_PATH         = f'{BASE_DIR}/Qdrant_db1'
ALIAS_PATH      = f'{BASE_DIR}/Qdrant_db1/alias_mapping.json'
MODEL_ID        = f'{BASE_DIR}/models/Qwen2.5-7B-Instruct-AWQ'
EXCEL_PATH      = f'{BASE_DIR}/example_data/example_data.xlsx'
API_CONFIG_PATH = f'{BASE_DIR}/API_config_data/Tài liệu config API.xlsx'   # Fixed path

COLLECTION_DOC = 'document_corpus'
COLLECTION_API = 'api_corpus'

import os, re, json, time, pickle, unicodedata, calendar
from typing import Optional, Dict, List, Any, Tuple
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

torch.set_grad_enabled(False)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if DEVICE=='cuda' else '-')

Device: cuda | GPU: Tesla T4


## 2. Load tất cả model & asset (1 lần lúc khởi động)

Pre-load để không tính giờ load lúc inference.

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM

t0 = time.time()
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
llm = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    attn_implementation='sdpa',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
llm.eval()
print(f'LLM loaded in {time.time()-t0:.1f}s')

# Pre-compute token id cho 'A','B','C','D' — dùng cho MCQA logprob
# Qwen tokenizer: 'A' và ' A' khác nhau. Lấy token id cuối khi encode chữ cái.
ABCD_IDS = {opt: tok.encode(opt, add_special_tokens=False)[-1] for opt in 'ABCD'}
print('ABCD token ids:', ABCD_IDS)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting PYTORCH_ALLOC_CONF='expandable_segments:True,max_split_size_mb:256,garbage_collection_threshold:0.7' for memory saving.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0
Transformers : 5.8.1
Torch        : 2.10.0+cu128
Triton       : 3.6.0


...

INFO  Kernel: Auto-selection: adding candidate `AwqExllamaV2Linear`            


INFO  Kernel: selected -> `AwqExllamaV2Linear`.                                


Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

INFO  gc.collect() reclaimed 6172 objects in 0.327s                            


LLM loaded in 211.4s
ABCD token ids: {'A': 32, 'B': 33, 'C': 34, 'D': 35}


In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient
import joblib

t0 = time.time()
bge = SentenceTransformer(BGE_PATH, device=DEVICE)
bge.max_seq_length = 512
qdrant = QdrantClient(path=DB_PATH)
svm = joblib.load(SVM_PATH)
print(f'BGE + Qdrant + SVM loaded in {time.time()-t0:.1f}s')

# Reranker: cross-encoder cho disambig endpoint + rerank doc chunks
t0 = time.time()
RERANKER_ID = 'BAAI/bge-reranker-v2-m3'
reranker = CrossEncoder(RERANKER_ID, device=DEVICE, max_length=512)
print(f'Reranker loaded in {time.time()-t0:.1f}s')

# Alias mapping JSON (đã có sẵn các group: priorityList, assetGroup, ...)
if os.path.exists(ALIAS_PATH):
    with open(ALIAS_PATH, 'r', encoding='utf-8') as f:
        old_alias_data = json.load(f)
    print('Alias groups:', list(old_alias_data.keys())[:10])
else:
    old_alias_data = {}
    print('No alias_mapping.json found.')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BGE + Qdrant + SVM loaded in 116.7s


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Reranker loaded in 29.1s
Alias groups: ['priorityList', 'assetGroup', 'dtmsClass', 'procurementType', 'dtmsType', 'isProbation', 'lcntOption', 'lcntOptionDoing', 'bidPlanType', 'lcntDomainType']


In [ ]:
# Build ALIAS_BY_GROUP từ alias_mapping.json — match CẢ key (tiếng Việt) lẫn value (mã API).
# Structure mỗi group: list of (keyword_lower, api_value).
# Quy tắc:
#  - Bỏ key/value thuần số ("0", "1") — match dễ bị false positive.
#  - Match cả "Thiết bị văn phòng" lẫn "TBVP" → cùng trả về value "TBVP".

# === DEBUG: in structure thật của alias_mapping.json (xem ngay cell này) ===
print('=== old_alias_data structure ===')
for gname, items in list(old_alias_data.items())[:3]:
    print(f'\nGroup: {gname}')
    print(f'  type: {type(items).__name__}')
    if isinstance(items, list):
        print(f'  len: {len(items)}')
        if items:
            print(f'  first item type: {type(items[0]).__name__}')
            print(f'  first item: {repr(items[0])[:200]}')
    elif isinstance(items, dict):
        print(f'  sample: {repr(list(items.items())[:3])[:200]}')
    elif isinstance(items, str):
        print(f'  preview: {items[:200]}')
print('=' * 50)

# === Defensive parsing: handle list[dict] / dict / string / list[tuple] / list[str] ===
def _extract_pairs_from_group(items):
    pairs = []
    if isinstance(items, str):
        try:
            items = json.loads(items)
        except Exception:
            return pairs
    if isinstance(items, dict):
        for k, v in items.items():
            pairs.append((str(k), v))
        return pairs
    if isinstance(items, list):
        for it in items:
            if isinstance(it, dict):
                k = it.get('key') or it.get('name') or it.get('label')
                v = it.get('value') or it.get('code') or it.get('val')
                if k is not None and v is not None:
                    pairs.append((str(k), v))
            elif isinstance(it, (list, tuple)) and len(it) >= 2:
                pairs.append((str(it[0]), it[1]))
            elif isinstance(it, str):
                try:
                    parsed = json.loads(it)
                    if isinstance(parsed, dict):
                        k = parsed.get('key'); v = parsed.get('value')
                        if k and v is not None:
                            pairs.append((str(k), v))
                except Exception:
                    pass
    return pairs

ALIAS_BY_GROUP = {}
for group_name, items in old_alias_data.items():
    raw_pairs = _extract_pairs_from_group(items)
    if not raw_pairs:
        continue
    seen_kw = set()
    pairs = []
    for k_raw, v_raw in raw_pairs:
        k_str = str(k_raw).strip()
        v_str = str(v_raw).strip() if not isinstance(v_raw, bool) else ''
        if not k_str:
            continue
        # Add key (tiếng Việt) — bỏ qua key thuần số / quá ngắn
        kw_k = k_str.lower()
        if kw_k and not kw_k.isdigit() and len(kw_k) >= 2 and kw_k not in seen_kw:
            pairs.append((kw_k, v_raw))
            seen_kw.add(kw_k)
        # Add value (mã API) làm alias — bỏ qua value thuần số / quá ngắn / trùng key
        kw_v = v_str.lower()
        if kw_v and not kw_v.isdigit() and kw_v != kw_k and len(kw_v) >= 2 and kw_v not in seen_kw:
            pairs.append((kw_v, v_raw))
            seen_kw.add(kw_v)
    if pairs:
        ALIAS_BY_GROUP[group_name] = pairs

print(f'\nALIAS_BY_GROUP: {len(ALIAS_BY_GROUP)} groups')
for g, pairs in list(ALIAS_BY_GROUP.items())[:6]:
    print(f'  {g}: {len(pairs)} aliases')
    for kw, val in pairs[:4]:
        print(f'      "{kw}" -> {val!r}')

# Backward compat: flat ALIAS_MAP
ALIAS_MAP = {kw: v for pairs in ALIAS_BY_GROUP.values() for kw, v in pairs}
print(f'\nFlat ALIAS_MAP: {len(ALIAS_MAP)} entries')

# === Verify api_corpus collection (giữ nguyên) ===
api_info = qdrant.get_collection(COLLECTION_API)
print(f'\napi_corpus: {api_info.points_count} points')
pts, _ = qdrant.scroll(COLLECTION_API, limit=1, with_payload=True)
sample = pts[0].payload
print('Payload keys:', list(sample.keys()))
cfg = sample.get('endpoint_config')
print(f'endpoint_config type: {type(cfg).__name__}')
if isinstance(cfg, dict):
    print('  → required_params:', [p.get('name') for p in cfg.get('required_params', [])])
    print('  → optional_params:', [p.get('name') for p in cfg.get('optional_params', [])])


=== old_alias_data structure ===

Group: priorityList
  type: list
  len: 5
  first item type: dict
  first item: {'key': 'Highest', 'value': 'Highest'}

Group: assetGroup
  type: list
  len: 6
  first item type: dict
  first item: {'key': 'Thiết bị văn phòng', 'value': 'TBVP'}

Group: dtmsClass
  type: list
  len: 6
  first item type: dict
  first item: {'key': 'Dự án Đầu tư hỗ trợ phát triển khoa học công nghệ', 'value': 'Dự án Đầu tư hỗ trợ phát triển KHCN'}

ALIAS_BY_GROUP: 22 groups
  priorityList: 5 aliases
      "highest" -> 'Highest'
      "high" -> 'High'
      "medium" -> 'Medium'
      "low" -> 'Low'
  assetGroup: 8 aliases
      "thiết bị văn phòng" -> 'TBVP'
      "tbvp" -> 'TBVP'
      "tài sản phục vụ sản xuất kinh doanh" -> 'Tài sản phục vụ sản xuất kinh doanh'
      "tài sản cố định" -> 'Tài sản cố định'
  dtmsClass: 8 aliases
      "dự án đầu tư hỗ trợ phát triển khoa học công nghệ" -> 'Dự án Đầu tư hỗ trợ phát triển KHCN'
      "dự án đầu tư hỗ trợ phát triển khcn" -

## 3. Layer 0 — Signal extraction (regex, không LLM)

Trích thực thể từ câu hỏi 1 lần, dùng cho cả router và handler.

In [ ]:
ORG_WHITELIST = ['TTPMVT', 'TTPMQT', 'TTPMTCS', 'TTPMCNM', 'TTPMCDS', 'TTCNDT']
PROJECT_TYPE_MAP = {
    'osdc': 'odc/osdc', 'odc': 'odc/osdc', 'odc/osdc': 'odc/osdc',
    't&m': 'T&M', 't & m': 'T&M', 'tm': 'T&M',
    'presales': 'presales', 'presale': 'presales',
    'package': 'package',
}
PROJECT_STATUS_MAP = {
    'in-progress': 'in-progress', 'đang triển khai': 'in-progress', 'đang thực hiện': 'in-progress',
    'hold': 'hold', 'tạm dừng': 'hold',
    'closed': 'closed', 'đã đóng': 'closed', 'kết thúc': 'closed',
    'open': 'open', 'mở mới': 'open',
}

def _last_day(y: int, m: int) -> str:
    d = calendar.monthrange(y, m)[1]
    return f'{y}-{m:02d}-{d:02d}'

def parse_time_range(q: str) -> Optional[Tuple[str, str]]:
    m = re.search(r'T(\d{1,2})\s*/\s*(\d{4})\s*[-→to>]+\s*T(\d{1,2})\s*/\s*(\d{4})', q, re.I)
    if m:
        m1, y1, m2, y2 = int(m[1]), int(m[2]), int(m[3]), int(m[4])
        return (f'{y1}-{m1:02d}-01', _last_day(y2, m2))
    m = re.search(r'tháng\s+(\d{1,2})\s*/\s*(\d{4})', q, re.I)
    if m:
        mo, y = int(m[1]), int(m[2])
        return (f'{y}-{mo:02d}-01', _last_day(y, mo))
    m = re.search(r'(?:quý|q)\s*(\d)\s*/\s*(\d{4})', q, re.I)
    if m:
        qn, y = int(m[1]), int(m[2])
        m1 = (qn - 1) * 3 + 1
        m2 = qn * 3
        return (f'{y}-{m1:02d}-01', _last_day(y, m2))
    m = re.search(r'T(\d{1,2})\s*/\s*(\d{4})', q, re.I)
    if m:
        mo, y = int(m[1]), int(m[2])
        return (f'{y}-{mo:02d}-01', _last_day(y, mo))
    m = re.search(r'năm\s+(\d{4})', q, re.I)
    if m:
        y = int(m[1])
        return (f'{y}-01-01', f'{y}-12-31')
    return None

def _match_alias_keyword(text_lower: str, keyword: str) -> bool:
    """Match alias trong text. Keyword dài (≥4 hoặc có space): in-substring.
       Keyword ngắn (mã viết tắt): word boundary ASCII."""
    if len(keyword) >= 4 or ' ' in keyword:
        return keyword in text_lower
    return bool(re.search(rf'(?<![a-z0-9]){re.escape(keyword)}(?![a-z0-9])', text_lower))

def extract_signals(q: str) -> Dict[str, Any]:
    sig = {
        'public_id': None, 'has_doc_phrase': False,
        'time_range': None, 'orgs': [], 'project_types': [],
        'project_status': [], 'is_company': False,
        'alias_by_group': {},  # {group_name: [api_values]}
    }
    m = re.search(r'Public[_\s]*(\d{1,4})', q, re.I)
    if m:
        sig['public_id'] = f'Public_{int(m[1]):03d}'
    if re.search(r'theo\s+tài\s+liệu|trong\s+tài\s+liệu|phụ\s+lục|trong\s+phụ\s+lục', q, re.I):
        sig['has_doc_phrase'] = True
    sig['time_range'] = parse_time_range(q)
    for org in ORG_WHITELIST:
        if re.search(rf'\b{org}\b', q, re.I):
            sig['orgs'].append(org)
    if re.search(r'cả\s+công\s+ty|toàn\s+công\s+ty|toàn\s+bộ\s+công\s+ty', q, re.I):
        sig['is_company'] = True
    ql = q.lower()
    for k, v in PROJECT_TYPE_MAP.items():
        if re.search(rf'(?<![a-z]){re.escape(k)}(?![a-z])', ql):
            if v not in sig['project_types']:
                sig['project_types'].append(v)
    for k, v in PROJECT_STATUS_MAP.items():
        if k in ql and v not in sig['project_status']:
            sig['project_status'].append(v)

    # Alias by group (match cả key tiếng Việt lẫn value mã API)
    for group_name, pairs in ALIAS_BY_GROUP.items():
        matched = []
        for kw, val in pairs:
            if _match_alias_keyword(ql, kw):
                if val not in matched:
                    matched.append(val)
        if matched:
            sig['alias_by_group'][group_name] = matched
    return sig

# Smoke test
for q in [
    'Trong T1/2025 - T12/2025 thì TTPMVT có bao nhiêu nhân sự',
    'Trong Quý 3/2025 thì TTPMTCS có bao nhiêu dự án',
    'Theo tài liệu Public_002, học phần Bảo mật Web?',
    'Trong T01/2025 -> T12/2025 thì với loại dự án là OSDC',
    'Danh sách tài sản nhóm TBVP của TTPMVT',                # value match
    'Danh sách tài sản thuộc Thiết bị văn phòng',            # key match
    'Mức độ ưu tiên Highest có bao nhiêu dự án',             # priority value
]:
    s = extract_signals(q)
    print(q)
    print('  orgs:', s['orgs'], '| types:', s['project_types'], '| alias:', s['alias_by_group'])
    print()

Trong T1/2025 - T12/2025 thì TTPMVT có bao nhiêu nhân sự
  orgs: ['TTPMVT'] | types: [] | alias: {'organization': ['TTPMVT']}

Trong Quý 3/2025 thì TTPMTCS có bao nhiêu dự án
  orgs: ['TTPMTCS'] | types: [] | alias: {'organization': ['TTPMTCS']}

Theo tài liệu Public_002, học phần Bảo mật Web?
  orgs: [] | types: [] | alias: {}

Trong T01/2025 -> T12/2025 thì với loại dự án là OSDC
  orgs: [] | types: ['odc/osdc'] | alias: {}

Danh sách tài sản nhóm TBVP của TTPMVT
  orgs: ['TTPMVT'] | types: [] | alias: {'assetGroup': ['TBVP'], 'organization': ['TTPMVT']}

Danh sách tài sản thuộc Thiết bị văn phòng
  orgs: [] | types: [] | alias: {'assetGroup': ['TBVP']}

Mức độ ưu tiên Highest có bao nhiêu dự án
  orgs: [] | types: [] | alias: {'priorityList': ['Highest', 'High']}



## 4. Layer 1 — Hybrid Router

1. Hard rule (Public_XXX, "Theo tài liệu", time+org)
2. SVM với BGE embedding (chỉ khi rule không quyết định được)
3. KHÔNG dùng LLM tiebreaker (tiết kiệm thời gian)

In [ ]:
# Cache query embedding để share giữa router và retrieval
def encode_query(q: str) -> np.ndarray:
    return bge.encode(q, normalize_embeddings=True, show_progress_bar=False)

def route(question: str, signals: Dict, q_vec: np.ndarray) -> Tuple[str, str, float]:
    """Return (route, method, confidence)."""
    # --- Tầng 1: hard rule ---
    if signals['public_id']:
        return 'call_document', 'rule:public_id', 1.0
    if signals['has_doc_phrase']:
        return 'call_document', 'rule:doc_phrase', 1.0
    if signals['time_range'] and (signals['orgs'] or signals['is_company'] or signals['project_types']):
        return 'call_api', 'rule:time+entity', 1.0
    # Câu có cụm điển hình API (chỉ số dashboard)
    if re.search(r'defect\s*rate|leakage\s*rate|slsx|slnt|free\s*effort|free\s*effort|tuyển\s*mới', question, re.I):
        return 'call_api', 'rule:api_keyword', 0.9

    # --- Tầng 2: SVM ---
    v = q_vec.reshape(1, -1)
    if hasattr(svm, 'predict_proba'):
        probs = svm.predict_proba(v)[0]
        # Class label thường: 0=document, 1=api (chỉnh nếu SVM của bạn ngược lại)
        classes = list(getattr(svm, 'classes_', [0, 1]))
        p_doc = probs[classes.index(0)] if 0 in classes else probs[0]
        p_api = probs[classes.index(1)] if 1 in classes else probs[1]
        if p_api >= p_doc:
            return 'call_api', 'svm', float(p_api)
        else:
            return 'call_document', 'svm', float(p_doc)
    pred = svm.predict(v)[0]
    return ('call_api' if pred == 1 else 'call_document'), 'svm-hard', 0.5

## 5. Layer 2a — call_document handler (MCQA via logprob)

**Trick chính**: thay vì để LLM generate text rồi parse, ta tính logprob của token "A"/"B"/"C"/"D" sau prompt. 1 forward pass, không bao giờ ra rác.

In [ ]:
def parse_options(note: str) -> Dict[str, str]:
    if not isinstance(note, str):
        return {}
    opts = {}
    for letter in 'ABCD':
        m = re.search(rf'(?:^|\n)\s*{letter}\s*[,.\)]\s*([^\n]+)', note)
        if m:
            v = m.group(1).strip()
            if v.lower() != 'nan':
                opts[letter] = v
    return opts

from qdrant_client.models import Filter, FieldCondition, MatchValue

def retrieve_doc_chunks(q_vec: np.ndarray, public_id: Optional[str], k: int = 10) -> List[str]:
    """Dense retrieve top-k chunks. Filter theo source_folder nếu có public_id."""
    flt = None
    if public_id:
        flt = Filter(must=[FieldCondition(key='source_folder', match=MatchValue(value=public_id))])
    try:
        hits = qdrant.query_points(
            collection_name=COLLECTION_DOC,
            query=q_vec.tolist(),
            query_filter=flt,
            limit=k,
        ).points
    except Exception as e:
        print('Qdrant doc search error:', e)
        return []
    chunks = []
    for h in hits:
        # Payload thật: 'content' (xem cell sanity check)
        txt = h.payload.get('content') or h.payload.get('text') or h.payload.get('chunk') or ''
        if txt:
            chunks.append(txt)
    return chunks

def rerank_chunks(question: str, chunks: List[str], top_k: int = 5) -> List[str]:
    if not chunks:
        return []
    if len(chunks) <= top_k:
        return chunks
    pairs = [[question, c] for c in chunks]
    scores = reranker.predict(pairs, show_progress_bar=False, batch_size=8)
    idx = np.argsort(-np.asarray(scores))[:top_k]
    return [chunks[i] for i in idx]

def is_multi_answer(question: str) -> bool:
    """Detect câu cần chọn nhiều đáp án."""
    return bool(re.search(
        r'các\s+(đáp\s*án|phương\s*án|lựa\s*chọn|ý)|tất\s+cả|những\s+(đáp\s*án|phương\s*án|ý)|chọn\s+(nhiều|tất\s*cả|các)',
        question, re.I
    ))

@torch.no_grad()
def mcqa_logprob(question: str, options: Dict[str, str], context: str = '', multi: bool = False) -> Tuple[List[str], Dict[str, float]]:
    """Pick A/B/C/D via logprob — 1 forward pass. Multi-answer hỗ trợ qua softmax threshold."""
    if not options:
        return ['A'], {}
    opt_lines = '\n'.join(f'{k}. {v}' for k, v in options.items())
    if context:
        user_prompt = (
            f'Dựa vào ngữ cảnh sau, chọn đáp án đúng nhất.\n\n'
            f'NGỮ CẢNH:\n{context}\n\n'
            f'CÂU HỎI: {question}\n\n'
            f'LỰA CHỌN:\n{opt_lines}\n\n'
            f'Đáp án đúng (chỉ in 1 ký tự A, B, C hoặc D):'
        )
    else:
        user_prompt = (
            f'Chọn đáp án đúng nhất cho câu hỏi.\n\n'
            f'CÂU HỎI: {question}\n\n'
            f'LỰA CHỌN:\n{opt_lines}\n\n'
            f'Đáp án đúng (chỉ in 1 ký tự A, B, C hoặc D):'
        )
    msgs = [{'role': 'user', 'content': user_prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    input_ids = tok(text, return_tensors='pt', truncation=True, max_length=2000).input_ids.to(llm.device)
    out = llm(input_ids=input_ids, use_cache=False)
    next_logits = out.logits[0, -1, :]

    # Softmax chỉ trên các option có sẵn
    valid_logits = torch.tensor([next_logits[ABCD_IDS[opt]].item() for opt in options.keys()])
    probs_t = torch.softmax(valid_logits, dim=0)
    probs = {opt: float(probs_t[i]) for i, opt in enumerate(options.keys())}

    if multi:
        picks = [opt for opt, p in probs.items() if p >= 0.25]
        if not picks:
            picks = [max(probs, key=probs.get)]
    else:
        picks = [max(probs, key=probs.get)]
    return picks, probs

def handle_document(question: str, note: Optional[str], signals: Dict, q_vec: np.ndarray) -> Dict:
    options = parse_options(note or '')
    if not options:
        return {'numbers': 1, 'result': 'A'}
    # Always retrieve (default ON, không skip nữa)
    chunks = retrieve_doc_chunks(q_vec, signals.get('public_id'), k=10)
    if chunks:
        top = rerank_chunks(question, chunks, top_k=5)
        ctx = '\n---\n'.join(top)[:3500]
    else:
        ctx = ''
    multi = is_multi_answer(question)
    picks, _ = mcqa_logprob(question, options, ctx, multi=multi)
    return {'numbers': len(picks), 'result': ','.join(sorted(picks))}

## 6. Layer 2b — call_api handler

1. Encode 131 API doc 1 lần lúc khởi động.
2. Query → top-3 endpoint bằng cosine similarity.
3. Param fill **deterministic-first**: time/org/projectType bằng regex+alias.
4. LLM chỉ generate body khi có param required còn thiếu — output JSON strict.

In [ ]:
# Đã chuyển sang dùng api_corpus Qdrant collection (đã verify ở cell trên).
# Không cần encode lại 131 endpoint từ xlsx — tiết kiệm ~5s startup + RAM.
# Retrieve sẽ qua qdrant.query_points (xem cell tiếp theo).
print('Dùng api_corpus Qdrant collection cho retrieval. Không encode lại.')

Dùng api_corpus Qdrant collection cho retrieval. Không encode lại.


In [ ]:
def _apply_hard_path_rules(question: str, candidates: List[Dict]) -> List[Dict]:
    """Re-prioritize candidates for known reranker disambiguation errors."""
    ql = question.lower()

    def prefer(path_key: str) -> List[Dict]:
        pk = path_key.lower()
        m = [c for c in candidates if pk in c.get("config", {}).get("request", {}).get("path", "").lower()]
        o = [c for c in candidates if c not in m]
        return (m + o) if m else candidates

    if re.search(r'tỷ\s*lệ\s+sử\s+dụng\s+nguồn\s+lực', ql):
        return prefer('get-compare-ru')
    if re.search(r'defect.*trong\s+k[ỳy]', ql) and 'lũy' not in ql:
        return prefer('all-curr')
    if 'free effort' in ql and re.search(r'ra\s+role', ql):
        return prefer('get-free-effort-bu')
    if 'slsx' in ql and re.search(r'presales|odc|osdc|package', ql) and not re.search(r'thực\s+hiện|kế\s+hoạch', ql):
        return prefer('pmo-027')
    return candidates


def retrieve_api_endpoint(question: str, q_vec: np.ndarray,
                          top_k_dense: int = 20, top_k_final: int = 3) -> List[Dict]:
    """Dense retrieve api_corpus top_k_dense, rerank với cross-encoder, trả top_k_final."""
    hits = qdrant.query_points(
        collection_name=COLLECTION_API,
        query=q_vec.tolist(),
        limit=top_k_dense,
    ).points

    candidates = []
    for h in hits:
        p = h.payload
        cfg = p.get("endpoint_config")
        if isinstance(cfg, str):
            try:
                cfg = json.loads(cfg)
            except Exception:
                cfg = {}
        text = f"{p.get('name','')}. {p.get('description','')} Ví dụ: {p.get('example_question','')}"
        candidates.append({
            'func_code': p.get('func_code'),
            'name': p.get('name', ''),
            'text': text,
            'config': cfg or {},
            'dense_score': float(h.score),
        })

    if not candidates:
        return []

    # Rerank với cross-encoder
    pairs = [[question, c['text']] for c in candidates]
    rerank_scores = reranker.predict(pairs, show_progress_bar=False, batch_size=8)
    for c, s in zip(candidates, rerank_scores):
        c['rerank_score'] = float(s)
    candidates.sort(key=lambda x: -x['rerank_score'])
    candidates = _apply_hard_path_rules(question, candidates)
    return candidates[:top_k_final]

In [ ]:
def _infer_type(time_range) -> int:
    """Map time_range to API type param (ra-report endpoints).
    Rules: single month->2, exact quarter->3, 4-11 months->2, full year->3."""
    if not time_range:
        return 3
    fy, fm = int(time_range[0][:4]), int(time_range[0][5:7])
    ty, tm = int(time_range[1][:4]), int(time_range[1][5:7])
    total_months = (ty - fy) * 12 + (tm - fm + 1)
    if total_months == 12:
        return 3   # full year -> MONTH granularity
    if total_months == 3 and fm in (1, 4, 7, 10):
        return 3   # exact quarter -> MONTH granularity
    if total_months == 1:
        return 2   # single month -> WEEK granularity
    return 2       # other range -> WEEK granularity


def default_value(ptype: str):
    pt = ptype.lower()
    if 'list' in pt:
        return []
    if 'bool' in pt:
        return False
    if 'int' in pt or 'long' in pt or 'decimal' in pt or 'number' in pt:
        return None
    if 'date' in pt:
        return None
    return None

def build_body_deterministic(cfg: Dict, signals: Dict, question: str) -> Tuple[Dict, List[str]]:
    """Fill body theo required + optional params bằng rule. Return (body, missing_required)."""
    body = {}
    missing = []
    req_params = cfg.get('required_params', []) or []
    opt_params = cfg.get('optional_params', []) or []
    all_params = req_params + opt_params
    required_names = {p['name'] for p in req_params}
    alias_groups = signals.get('alias_by_group', {})

    for p in all_params:
        name = p.get('name')
        ptype = p.get('type', '')
        val = default_value(ptype)

        if name in ('fromDate', 'toDate'):
            if signals['time_range']:
                val = signals['time_range'][0] if name == 'fromDate' else signals['time_range'][1]
        elif name in ('fromDateProject', 'toDateProject'):
            val = None
        elif name == 'type':
            val = _infer_type(signals.get('time_range'))
        elif name == 'organization':
            val = signals['orgs'] if signals['orgs'] else []
        elif name == 'projectType':
            val = signals['project_types'] if signals['project_types'] else []
        elif name == 'projectStatus':
            val = signals['project_status'] if signals['project_status'] else []
        elif name in ('projectList', 'customerList'):
            val = []
        elif name == 'isCompany':
            val = signals['is_company'] or (not bool(signals['orgs']))
        elif name and name.lower().startswith('isall'):
            val = True
        elif name == 'sort':
            _ql = question.lower()
            _desc_kw = ['ranking', 'xếp hạng', 'giảm dần', 'cao nhất', 'lọn nhất']
            _asc_kw  = ['tăng dần', 'thấp nhất', 'nhỏ nhất']
            if any(k in _ql for k in _desc_kw):
                val = 2
            elif any(k in _ql for k in _asc_kw):
                val = 1
            else:
                val = default_value(ptype)
        elif name == 'page':
            val = 0
        elif name == 'size':
            val = 100
        elif name == 'standardComparison':
            val = None
        elif name in alias_groups:
            # Param name khớp tên group trong alias_mapping -> dùng value đã match
            matches = alias_groups[name]
            if 'list' in ptype.lower():
                val = matches
            elif matches:
                val = matches[0]

        body[name] = val
        # Track required date còn thiếu
        if name in required_names and name in ('fromDate', 'toDate') and val is None:
            missing.append(name)

    return body, missing

@torch.no_grad()
def llm_fill_missing(question: str, cfg: Dict, body: Dict, missing: List[str]) -> Dict:
    """Chỉ gọi LLM để fill required param còn thiếu sau rule. Hiếm khi cần."""
    if not missing:
        return body
    schema_lines = []
    for p in (cfg.get('required_params', []) or []) + (cfg.get('optional_params', []) or []):
        if p['name'] in missing:
            schema_lines.append(f"- {p['name']} ({p.get('type','?')}): {p.get('description','')}")
    schema_str = '\n'.join(schema_lines)
    user_prompt = (
        f'Trích xuất các tham số còn thiếu từ câu hỏi. Trả về JSON duy nhất, không giải thích.\n\n'
        f'CÂU HỎi: {question}\n\n'
        f'CÁC THAM SỐ CẦN TRÌCH:\n{schema_str}\n\n'
        f'Trả về JSON với đúng các key trên. Ví dụ: {{"fromDate": "2025-01-01"}}'
    )
    msgs = [{'role': 'user', 'content': user_prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    input_ids = tok(text, return_tensors='pt', truncation=True, max_length=1500).input_ids.to(llm.device)
    out = llm.generate(
        input_ids, max_new_tokens=120, do_sample=False, temperature=0.0,
        pad_token_id=tok.eos_token_id, use_cache=True,
    )
    gen = tok.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r'\{[^{}]*\}', gen, re.S)
    if not m:
        return body
    try:
        patch = json.loads(m.group(0))
        for k, v in patch.items():
            if k in missing:
                body[k] = v
    except Exception:
        pass
    return body

def handle_api(question: str, signals: Dict, q_vec: np.ndarray) -> Dict:
    candidates = retrieve_api_endpoint(question, q_vec, top_k_dense=20, top_k_final=3)
    if not candidates:
        return {'path': '', 'body': {}}
    best = candidates[0]
    cfg = best['config']
    path = cfg.get('request', {}).get('path', '')
    body, missing = build_body_deterministic(cfg, signals, question)
    if missing:
        body = llm_fill_missing(question, cfg, body, missing)

    # ── POST-PROCESSING ──
    # 1. Normalize IsAllCustomer casing (config uses capital-I, API expects lowercase-i)
    if 'IsAllCustomer' in body:
        body['isAllCustomer'] = body.pop('IsAllCustomer')

    # 2. fromDateProject / toDateProject: hidden params present in actual API but absent
    #    from Excel config. Endpoints with company-wide filter params (isCompany /
    #    IsAllCustomer / isAllProject) or with sort in required params always need them.
    opt_names = {p.get('name', '') for p in (cfg.get('optional_params', []) or [])}
    req_names  = {p.get('name', '') for p in (cfg.get('required_params', []) or [])}
    _p = path.lower()
    needs_fdp = (
        ('leakage-rate' in _p or 'defect-rate' in _p)
        and (
            bool(opt_names & {'isCompany', 'IsAllCustomer', 'isAllProject'})
            or 'sort' in req_names
        )
    )
    if needs_fdp:
        body.setdefault('fromDateProject', None)
        body.setdefault('toDateProject', None)
        body.setdefault('standardComparison', None)

    # 3. type for recruitment & employee-info/organization (absent from Excel config)
    if 'recruitment/new-organization' in path:
        body.setdefault('type', 4)
    elif 'recruitment/new-level' in path:
        body.setdefault('type', 2)
    elif 'recruitment/new-role' in path:
        body.setdefault('type', 2)
    elif 'employee-info/level' in path or 'employee-info/soc' in path or 'employee-info/dilution-ranking' in path:
        tr = signals.get('time_range')
        if tr:
            fy2, fm2 = int(tr[0][:4]), int(tr[0][5:7])
            ty2, tm2 = int(tr[1][:4]), int(tr[1][5:7])
            nm = (ty2 - fy2) * 12 + (tm2 - fm2 + 1)
            t = 5 if nm == 12 else (4 if nm == 3 and fm2 in (1, 4, 7, 10) else 3)
        else:
            t = 5
        body.setdefault('type', t)
        if 'employee-info/soc' in path:
            body.setdefault('organization', [])
        if 'employee-info/dilution-ranking' in path:
            body.setdefault('isCompany', not bool(signals.get('orgs')))
    elif 'employee-info/organization' in path:
        tr = signals.get('time_range')
        if tr:
            fy2, fm2 = int(tr[0][:4]), int(tr[0][5:7])
            ty2, tm2 = int(tr[1][:4]), int(tr[1][5:7])
            nm = (ty2 - fy2) * 12 + (tm2 - fm2 + 1)
            t = 5 if nm == 12 else (4 if nm == 3 and fm2 in (1, 4, 7, 10) else 3)
        else:
            t = 5
        body.setdefault('type', t)

    return {'path': path, 'body': body, '_func_code': best['func_code']}

## 7. Main pipeline + smoke test

In [ ]:
def pipeline(question: str, note: Optional[str] = None, debug: bool = False) -> Dict:
    t_total = time.time()
    timings = {}

    t0 = time.time()
    signals = extract_signals(question)
    timings['signals'] = time.time() - t0

    t0 = time.time()
    q_vec = encode_query(question)
    timings['encode'] = time.time() - t0

    t0 = time.time()
    route_label, route_method, route_conf = route(question, signals, q_vec)
    timings['route'] = time.time() - t0

    t0 = time.time()
    if route_label == 'call_document':
        param = handle_document(question, note, signals, q_vec)
    else:
        param = handle_api(question, signals, q_vec)
    timings['handle'] = time.time() - t0
    timings['total'] = time.time() - t_total

    out = {'func_code': route_label, 'func_param': param,
           'route_method': route_method, 'route_conf': route_conf, 'time': timings}
    if debug:
        out['signals'] = signals
    return out

# Smoke test
test_qs = [
    ('Trong T1/2025 - T12/2025 thì TTPMVT có RA thực hiện và kế hoạch là bao nhiêu MM.', None),
    ('Theo tài liệu Public_002, học phần Bảo mật Web trang bị kiến thức về bộ rủi ro phổ biến nào?',
     'A, OWASP Top 10\n B, IEEE 802.11\n C, ISO/IEC 27001\n D, NIST Cybersecurity Framework'),
    ('Một pixel của ảnh xám có thể nhận giá trị tối thiểu và tối đa lần lượt là?',
     'A, 0 và 127\n B, -127 và 127\n C, 0 và 255\n D, 0 và 1048576'),
]
for q, n in test_qs:
    r = pipeline(q, n, debug=False)
    print(f'Q: {q[:80]}')
    print(f'  -> {r["func_code"]} via {r["route_method"]} ({r["route_conf"]:.2f})')
    print(f'  param: {json.dumps(r["func_param"], ensure_ascii=False)[:200]}')
    print(f'  time:  {r["time"]["total"]:.2f}s {r["time"]}')
    print()

Q: Trong T1/2025 - T12/2025 thì TTPMVT có RA thực hiện và kế hoạch là bao nhiêu MM.
  -> call_api via rule:time+entity (1.00)
  param: {"path": "/api/v1/dashboard/project-performance/PMO-033", "body": {"fromDate": "2025-01-01", "toDate": "2025-12-31", "page": 0, "size": 100, "organization": ["TTPMVT"], "projectList": [], "projectType
  time:  2.04s {'signals': 0.0004496574401855469, 'encode': 1.130845069885254, 'route': 5.4836273193359375e-06, 'handle': 0.9082098007202148, 'total': 2.0395169258117676}

Q: Theo tài liệu Public_002, học phần Bảo mật Web trang bị kiến thức về bộ rủi ro p
  -> call_document via rule:public_id (1.00)
  param: {"numbers": 1, "result": "A"}
  time:  3.18s {'signals': 0.0004215240478515625, 'encode': 0.059602975845336914, 'route': 3.0994415283203125e-06, 'handle': 3.117403745651245, 'total': 3.177438259124756}

Q: Một pixel của ảnh xám có thể nhận giá trị tối thiểu và tối đa lần lượt là?
  -> call_document via svm (1.00)
  param: {"numbers": 1, "result": "C"}


## 8. Evaluation trên example_data.xlsx

Tính metric riêng cho:
- Router accuracy
- call_document accuracy (chỉ tính câu route đúng)
- call_api: path accuracy + body match
- Time mean / p50 / p95 / p99

In [ ]:
qdf = pd.read_excel(EXCEL_PATH, sheet_name='example_question')
rdf = pd.read_excel(EXCEL_PATH, sheet_name='example_result')
merged = qdf.merge(rdf, on='id')
print(f'Total: {len(merged)} câu')
print(f'Distribution: {merged["func_code"].value_counts().to_dict()}')

# Frontend params không tính vào item_accuracy theo luật chấm
FRONTEND_PARAMS = {'sort', 'page', 'size', 'pageSize', 'pageNumber', 'pagesize'}

def normalize_param_for_compare(p):
    if isinstance(p, dict):
        return p
    if not isinstance(p, str):
        return {}
    try:
        return json.loads(p)
    except Exception:
        pass
    try:
        fixed = re.sub(r'"body"\s*:\s*"({\[^}]*?\})"', r'"body": \1', p)
        return json.loads(fixed)
    except Exception:
        pass
    m = re.search(r'"path"\s*:\s*"([^"]+)"', p)
    if m:
        bm = re.search(r'"body"\s*:\s*"?(\{[\s\S]*?\})', p)
        body = {}
        if bm:
            try:
                body = json.loads(bm.group(1))
            except Exception:
                pass
        return {'path': m.group(1), 'body': body}
    return {}

def normalize_value(v):
    if isinstance(v, list):
        return tuple(sorted([str(x).lower().strip() for x in v]))
    if isinstance(v, str):
        return v.lower().strip()
    return v

def fmt_val(v, max_len=40):
    s = json.dumps(v, ensure_ascii=False) if not isinstance(v, str) else v
    return s if len(s) <= max_len else s[:max_len-3] + "..."

def score_api_with_diff(pred, gt):
    """Return (score, diff_str, n_correct, n_total).
    diff_str: chuỗi mô tả các field sai, format: 'field=gt|pred; ...'
    """
    pred_path = (pred.get('path') or '').strip()
    gt_path = (gt.get('path') or '').strip()
    if not gt_path or pred_path != gt_path:
        return 0.0, f'PATH_MISMATCH: gt={gt_path} pred={pred_path}', 0, 0

    gt_body = gt.get('body', {}) or {}
    if isinstance(gt_body, str):
        try:
            gt_body = json.loads(gt_body)
        except Exception:
            gt_body = {}
    pred_body = pred.get('body', {}) or {}

    fields = [k for k in gt_body.keys() if k not in FRONTEND_PARAMS]
    if not fields:
        return 1.0, '', 0, 0

    diffs = []
    missing_in_pred = []
    extra_in_pred = [k for k in pred_body.keys() if k not in gt_body and k not in FRONTEND_PARAMS]
    correct = 0
    for k in fields:
        gv = gt_body.get(k)
        pv = pred_body.get(k, '<MISSING>')
        if k not in pred_body:
            missing_in_pred.append(k)
            diffs.append(f'{k}=[{fmt_val(gv)}|MISSING]')
        elif normalize_value(pv) == normalize_value(gv):
            correct += 1
        else:
            diffs.append(f'{k}=[{fmt_val(gv)}|{fmt_val(pv)}]')
    extra_str = (' +EXTRA:' + ','.join(extra_in_pred)) if extra_in_pred else ''
    diff_str = ('; '.join(diffs) + extra_str).strip('; ')
    return correct / len(fields), diff_str, correct, len(fields)

def score_document(pred, gt):
    pr = str(pred.get('result', '')).upper()
    gr = str(gt.get('result', '')).upper()
    if not gr:
        return 0.0
    pred_set = set(s.strip() for s in pr.split(',') if s.strip())
    gt_set = set(s.strip() for s in gr.split(',') if s.strip())
    if not gt_set:
        return 0.0
    common = pred_set & gt_set
    extra = pred_set - gt_set
    return max(0.0, (len(common) - 0.5 * len(extra)) / len(gt_set))

def time_score(t):
    if t is None or t < 0:
        return 0.0
    if t <= 2.0:
        return 150.0
    if t >= 30.0:
        return 30.0
    return 150.0 - (t - 2.0) * (150.0 - 30.0) / (30.0 - 2.0)

rows = []
for _, row in tqdm(merged.iterrows(), total=len(merged)):
    note = row['note'] if pd.notna(row.get('note')) else None
    try:
        pred = pipeline(row['fun_question'], note=note)
    except Exception as e:
        print(f'Error on id={row["id"]}:', e)
        pred = {'func_code': 'call_document', 'func_param': {'numbers': 1, 'result': 'A'},
                'time': {'total': 30.0}, 'route_method': 'err', 'route_conf': 0}

    gt = normalize_param_for_compare(row['func_param'])
    route_ok = pred['func_code'] == row['func_code']

    body_diff = ''
    n_correct, n_total = 0, 0
    if not route_ok:
        item_acc = 0.0
        body_diff = f'ROUTE_MISMATCH: gt={row["func_code"]} pred={pred["func_code"]}'
    elif pred['func_code'] == 'call_document':
        item_acc = score_document(pred['func_param'], gt)
    else:
        item_acc, body_diff, n_correct, n_total = score_api_with_diff(pred['func_param'], gt)

    ts = time_score(pred['time']['total'])
    q_score = item_acc * ts

    rows.append({
        'id': row['id'],
        'gt_code': row['func_code'],
        'pred_code': pred['func_code'],
        'route_method': pred.get('route_method'),
        'route_ok': route_ok,
        'item_acc': item_acc,
        'all_ok': route_ok and item_acc >= 0.999,
        'time_s': pred['time']['total'],
        'time_score': ts,
        'q_score': q_score,
        'gt_path': gt.get('path') if isinstance(gt, dict) else '',
        'pred_path': pred['func_param'].get('path', '') if pred['func_code'] == 'call_api' else '',
        'gt_result': gt.get('result') if isinstance(gt, dict) else '',
        'pred_result': pred['func_param'].get('result', '') if pred['func_code'] == 'call_document' else '',
        'body_fields_correct': f'{n_correct}/{n_total}' if pred['func_code'] == 'call_api' else '',
        'body_diff': body_diff,
    })

ev = pd.DataFrame(rows)
print('\n========== METRICS (theo luật chấm) ==========')
print(f'Router acc:       {ev["route_ok"].mean()*100:.1f}%')
print(f'Full match acc:   {ev["all_ok"].mean()*100:.1f}%   (item_acc=1.0)')
print(f'Mean item_acc:    {ev["item_acc"].mean():.3f}   (avg partial credit)')
print(f'\nPer branch:')
print(ev.groupby('gt_code').agg(
    n=('id','count'),
    route_ok=('route_ok','mean'),
    item_acc=('item_acc','mean'),
    all_ok=('all_ok','mean'),
    time_mean=('time_s','mean'),
).round(3))
print(f'\nTime:  mean={ev["time_s"].mean():.2f}s  p50={ev["time_s"].median():.2f}s  p95={ev["time_s"].quantile(0.95):.2f}s')
print(f'Time score mean:        {ev["time_score"].mean():.1f}/150')
print(f'Mean q_score (item × time): {ev["q_score"].mean():.1f}')

RESULT_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(RESULT_DIR, exist_ok=True)

eval_csv_path = os.path.join(RESULT_DIR, 'eval_v91.csv')
ev.to_csv(eval_csv_path, index=False)
print(f'\nSaved eval to {eval_csv_path}')

# ====== Body diff analysis cho call_api ======
print('\n========== TOP 30 CÂU CALL_API SAI BODY ==========')
api_bad = ev[(ev["gt_code"]=="call_api") & (ev["item_acc"] < 1.0)].sort_values("item_acc")
for _, r in api_bad.head(30).iterrows():
    q = merged[merged["id"]==r["id"]]["fun_question"].iloc[0]
    print(f"id={r['id']}  item_acc={r['item_acc']:.2f}  ({r['body_fields_correct']})")
    print(f"  Q: {q[:100]}")
    print(f"  diff: {r['body_diff'][:300]}")
    print()

# ====== Phân tích field nào sai nhiều nhất ======
print('\n========== FIELD HAY SAI NHẤT ==========')
from collections import Counter
field_counter = Counter()
for _, r in api_bad.iterrows():
    diff = r["body_diff"]
    if not isinstance(diff, str) or not diff:
        continue
    for token in diff.split(";"):
        m = re.match(r"\s*([A-Za-z][A-Za-z0-9_]*)=", token)
        if m:
            field_counter[m.group(1)] += 1
for fname, cnt in field_counter.most_common(15):
    print(f"  {fname:25} sai {cnt} lần")

Total: 100 câu
Distribution: {'call_api': 50, 'call_document': 50}


  0%|          | 0/100 [00:00<?, ?it/s]


========== METRICS (theo luật chấm) ==========
Router acc:       100.0%
Full match acc:   90.0%   (item_acc=1.0)
Mean item_acc:    0.943   (avg partial credit)

Per branch:
                n  route_ok  item_acc  all_ok  time_mean
gt_code                                                 
call_api       50       1.0     0.987     0.9      0.662
call_document  50       1.0     0.900     0.9      2.130

Time:  mean=1.40s  p50=1.12s  p95=2.53s
Time score mean:        149.5/150
Mean q_score (item × time): 141.1

Saved eval to /content/drive/MyDrive/Workspace_Chatbot_LLM_Output/results/eval_v91.csv

========== TOP 30 CÂU CALL_API SAI BODY ==========
id=1090  item_acc=0.80  (8/10)
  Q: Trong T10/2025, tôi muốn xem các thông số leakage rate trong kỳ của dự án BU05.VTT.MyViettel có các 
  diff: toDate=[2025-10-31|MISSING]; projectList=[[11903]|[]] +EXTRA:Tên trường,isCompany,isAllProject,isAllCustomer

id=1105  item_acc=0.83  (5/6)
  Q: Trong T11/2025, tôi muốn xem ở TTPMVT, tỉ lệ pha loãng theo

In [ ]:
print('=== Câu router SAI ===')
for _, r in ev[~ev['route_ok']].head(20).iterrows():
    q = merged[merged['id']==r['id']]['fun_question'].iloc[0]
    print(f"id={r['id']}  gt={r['gt_code']} pred={r['pred_code']} via {r['route_method']}")
    print(f'  Q: {q[:120]}')
    print()

print('\n=== call_document SAI (router đúng nhưng param sai) ===')
doc_wrong = ev[(ev['gt_code']=='call_document') & ev['route_ok'] & (ev['item_acc'] < 1.0)]
for _, r in doc_wrong.head(10).iterrows():
    q = merged[merged['id']==r['id']]['fun_question'].iloc[0]
    print(f"id={r['id']}  gt={r['gt_result']} pred={r['pred_result']}")
    print(f'  Q: {q[:120]}')

print('\n=== call_api SAI path ===')
api_wrong = ev[(ev['gt_code']=='call_api') & ev['route_ok'] & (ev['item_acc'] < 1.0)]
for _, r in api_wrong.head(10).iterrows():
    q = merged[merged['id']==r['id']]['fun_question'].iloc[0]
    print(f"id={r['id']}")
    print(f'  gt:   {r["gt_path"]}')
    print(f'  pred: {r["pred_path"]}')
    print(f'  Q: {q[:120]}')
    print()

=== Câu router SAI ===

=== call_document SAI (router đúng nhưng param sai) ===
id=714  gt=A pred=B
  Q: Tác dụng bất lợi và chống chỉ định khi sử dụng thiết bị AirStart 10 ?
id=719  gt=C pred=B
  Q: Trong Phụ lục II, tiêu chí nào xác định tính toàn vẹn của thông điệp dữ liệu đã ký số?
id=729  gt=C pred=D
  Q: Với hash `0160375e19e606d06f672be6e43f70fa70093d2a30031affd2929a5c446d07c1`, URL nào đã được tệp tương ứng truy cập vào 
id=730  gt=A pred=B
  Q: Đâu là cấu trúc heap ta nên dùng nếu muốn sắp xếp mảng giảm dần?
id=738  gt=C pred=B
  Q: Trong heap, chỉ số node con trái và phải của node x lần lượt là? (node gốc có chỉ số 0)

=== call_api SAI path ===
id=1090
  gt:   /api/v1/dashboard/leakage-rate/QA-038
  pred: /api/v1/dashboard/leakage-rate/QA-038
  Q: Trong T10/2025, tôi muốn xem các thông số leakage rate trong kỳ của dự án BU05.VTT.MyViettel có các giá trị như nào

id=1063
  gt:   /api/v1/dashboard/defect-rate/all-cum
  pred: /api/v1/dashboard/defect-rate/all-cum
  Q: Trong T11/

In [ ]:
## 11. Xuất kết quả example_data → submission CSV

import os, json, time
import pandas as pd
from tqdm.auto import tqdm

RESULT_DIR = f'{BASE_DIR}/results'
os.makedirs(RESULT_DIR, exist_ok=True)

def fmt_func_param(func_code: str, param: dict) -> str:
    """Format func_param đúng chuẩn submission."""
    if func_code == 'call_api':
        out = {
            'path': param.get('path', ''),
            'body': {k: v for k, v in param.get('body', {}).items() if k != '_func_code'}
        }
    else:
        out = {
            'numbers': param.get('numbers', 1),
            'result':  param.get('result', 'A')
        }
    return json.dumps(out, ensure_ascii=False, indent=2)

# Load example questions
qdf = pd.read_excel(EXCEL_PATH, sheet_name='example_question')
print(f'Example questions: {len(qdf)} câu')

rows = []
for _, row in tqdm(qdf.iterrows(), total=len(qdf), desc='Running example'):
    qid      = int(row['id'])
    question = str(row['fun_question'])
    note     = str(row['note']) if pd.notna(row.get('note')) else None

    t0  = time.time()
    res = pipeline(question, note)
    elapsed = time.time() - t0

    rows.append({
        'id':            qid,
        'func_code':     res['func_code'],
        'func_param':    fmt_func_param(res['func_code'], res['func_param']),
        'time_response': round(elapsed, 3),
    })

example_csv = f'{RESULT_DIR}/submission_example.csv'
pd.DataFrame(rows).to_csv(example_csv, index=False, encoding='utf-8-sig')
print(f'Saved {len(rows)} rows -> {example_csv}')

Example questions: 100 câu


Running example:   0%|          | 0/100 [00:00<?, ?it/s]

Saved 100 rows -> /content/drive/MyDrive/Workspace_Chatbot_LLM_Output/result/submission_example.csv


In [ ]:
## 12. Xuất kết quả test data → submission CSV

# Đổi TEST_PATH về đúng file test của cuộc thi
TEST_PATH = f'{BASE_DIR}/Test_data/Test_data.xlsx'

import os, json, time
import pandas as pd
from tqdm.auto import tqdm

RESULT_DIR = f'{BASE_DIR}/results'
os.makedirs(RESULT_DIR, exist_ok=True)

if not os.path.exists(TEST_PATH):
    print(f'[ERROR] Không tìm thấy test file: {TEST_PATH}')
    print('Hãy đặt lại TEST_PATH về đúng file câu hỏi test.')
else:
    tdf = pd.read_excel(TEST_PATH)
    # Hỗ trợ cả tên cột fun_question lẫn question
    q_col = 'fun_question' if 'fun_question' in tdf.columns else 'question'
    n_col = 'note' if 'note' in tdf.columns else None
    print(f'Test questions: {len(tdf)} câu | columns: {list(tdf.columns)}')

    rows = []
    for _, row in tqdm(tdf.iterrows(), total=len(tdf), desc='Running test'):
        qid      = int(row['id'])
        question = str(row[q_col])
        note     = str(row[n_col]) if n_col and pd.notna(row.get(n_col)) else None

        t0  = time.time()
        res = pipeline(question, note)
        elapsed = time.time() - t0

        rows.append({
            'id':            qid,
            'func_code':     res['func_code'],
            'func_param':    fmt_func_param(res['func_code'], res['func_param']),
            'time_response': round(elapsed, 3),
        })

    test_csv = f'{RESULT_DIR}/submission_test.csv'
    pd.DataFrame(rows).to_csv(test_csv, index=False, encoding='utf-8-sig')
    print(f'Saved {len(rows)} rows -> {test_csv}')

Test questions: 617 câu | columns: ['id', 'fun_question', 'note']


Running test:   0%|          | 0/617 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Saved 617 rows -> /content/drive/MyDrive/Workspace_Chatbot_LLM_Output/result/submission_test.csv


## 9. Sanity check — kiểm tra Qdrant payload schema

Chạy nếu retrieval không trả kết quả gì (có thể tên field payload khác).

In [19]:
info = qdrant.get_collection(COLLECTION_DOC)
print('Collection info:', info)
# Lấy 3 point bất kỳ để xem payload
pts, _ = qdrant.scroll(collection_name=COLLECTION_DOC, limit=3, with_payload=True)
for p in pts:
    print('Payload keys:', list(p.payload.keys()))
    for k, v in p.payload.items():
        print(f'  {k}: {str(v)[:120]}')
    print()

Collection info: status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=9508 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optimization_threads=1, prevent_unoptimized=N

## 10. TODO sau khi chạy lần đầu

Dựa vào output cell 8 (metrics + per-branch + debug list):

1. Nếu **Qdrant payload field name khác** (`source_folder`/`text`) → sửa trong `retrieve_doc()`.
2. Nếu **SVM class label ngược** (0=api thay vì 0=doc) → sửa trong `route()`.
3. Nếu **router acc < 90%** → xem các câu route sai, thêm rule.
4. Nếu **call_document no-context (general knowledge) sai nhiều** → có thể model 7B chưa đủ, thử cloze scoring full option text thay vì logprob 1 token.
5. Nếu **call_api sai path nhiều** → bật reranker bge-reranker-base trên top-20 retrieval.
6. Nếu **time/câu > 10s** → giảm `max_new_tokens` của `llm_fill_missing`, hoặc bỏ retrieval cho call_document no-context (đã làm).
7. Nếu **mọi thứ ổn nhưng vẫn chậm** → convert Qwen sang EXL2 4bpw + dùng ExLlamaV2 (tăng ~2× speed).